# Notebook 02 - Yelp Data Preprocessing

This notebook specifically focuses on the data preprocessing code to help us filter our datasets before they are split for downstream use in training and evaluation. This part of the pipeline also includes our pick for k-core, filtering by open restaurants, features for our towers and feature-engineered choices, generation of some embeddings, and splitting before save.

### Filtering of Business Data

In [1]:
import pandas as pd
import json
from google.colab import drive

drive.mount("/content/drive")
# yelp_business_data_path = "../Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json"
yelp_business_data_path = "/content/drive/MyDrive/rec_system/yelp_academic_dataset_business.json"
with open(yelp_business_data_path, "r") as f:
    yelp_business_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to open restaurants/food businesses only
mask = (
    yelp_business_df["categories"].str.contains("Restaurants|Food", na=False) &
    (yelp_business_df["is_open"] == 1)
)
yelp_open_restaurants_df = yelp_business_df[mask].copy()

print(f"{len(yelp_open_restaurants_df):} open restaurant/food businesses")

Mounted at /content/drive
44582 open restaurant/food businesses


### Filtration of Reviews by Business ID

In [2]:
yelp_review_data_path = "/content/drive/MyDrive/rec_system/yelp_academic_dataset_review.json"
with open(yelp_review_data_path, "r") as f:
    yelp_reviews_df = pd.DataFrame(json.loads(line) for line in f)

# Filter reviews to open restaurants only
open_restaurant_ids = set(yelp_open_restaurants_df["business_id"])
yelp_reviews_df = yelp_reviews_df[yelp_reviews_df["business_id"].isin(open_restaurant_ids)].copy()

print(f"{len(yelp_reviews_df):} reviews for open restaurants")

4104995 reviews for open restaurants


### K-Core Filtering

From the EDA notebook, we got the following table of values for the k-core statistics.  

| k-value | # of users | # of reviews | avg # of reviews/user|
| --- | --- | --- | --- |
| k = 5 | 210,133 users | 3,149,767 reviews | 15.0 avg reviews/user |
| k = 6 | 166,691 users | 2,932,557 reviews | 17.6 avg reviews/user |
| k = 7 | 136,534 users | 2,751,615 reviews | 20.2 avg reviews/user |

Based on the data, we decided to pick k = 5 as our k-core value. The reason behind picking k = 5  
was because we wanted to have a balanced tradeoff between the number of users, number of reviews,  
and the roughly average number of reviews/user.   

With a minimum baseline of 5 reviews for each user, this would also help combat reviews from low-activity users  
from affecting our recommendations significantly, ensuring that the quality of recommendations is up to a higher standard.

We are applying iterative k-core filtering to ensure every user has at least 5 reviews AND every business has at least 5 reviews simultaneously. Although the business JSON had a minimum of 5 reviews, after filtering to open restaurants only, some businesses lost reviews and dropped below k = 5. As previously mentioned in EDA, we expect to see slightly lower numbers than what the table states.

In [3]:
k = 5
while True:
    user_counts = yelp_reviews_df["user_id"].value_counts()
    business_counts = yelp_reviews_df["business_id"].value_counts()

    val_users = set(user_counts[user_counts >= k].index)
    val_businesses = set(business_counts[business_counts >= k].index)

    k_filter_mask = yelp_reviews_df[
        yelp_reviews_df["user_id"].isin(val_users) &
        yelp_reviews_df["business_id"].isin(val_businesses)
    ]

    if len(k_filter_mask) == len(yelp_reviews_df):
        break

    yelp_reviews_df = k_filter_mask.copy()

print(f"{yelp_reviews_df['user_id'].nunique():} users, {yelp_reviews_df['business_id'].nunique():} businesses, {len(yelp_reviews_df):} reviews after k-core filtering")

166488 users, 39140 businesses, 2323580 reviews after k-core filtering


As evidenced in the results above, the number of users, businesses, and reviews in consideration reduced.  
However, roughly 166K users, 39K businesses, and 2.3M reviews is sufficient for creating meaningful training, validation, and test splits.

### Filtering of Users (Matching with Reviews User IDs)

In [4]:
yelp_users_data_path = "/content/drive/MyDrive/rec_system/yelp_academic_dataset_user.json"
with open(yelp_users_data_path, "r") as f:
    yelp_users_df = pd.DataFrame(json.loads(line) for line in f)

# Filter to only users present in k-core filtered reviews
review_user_ids = set(yelp_reviews_df["user_id"])
yelp_users_df = yelp_users_df[yelp_users_df["user_id"].isin(review_user_ids)].copy()

# Checking if the user count matches
assert len(yelp_users_df) == yelp_reviews_df["user_id"].nunique(), "User count mismatch!"
print(f"User count matches: {len(yelp_users_df):} users")

User count matches: 166488 users


### Feature Engineering

We added filtering logic above to handle the filtering of the three key datasets for our systems. Now, we'll work on the feature engineering to curate the features that will be helpful for generating proper recommendations.

#### User Tower - feature 1: avg_stars_given

In [5]:
# avg_stars_given is now computed AFTER the temporal split from train rows
# only (see post-split cell). The raw user JSON's `average_stars` is a
# snapshot at extraction time across the user's entire Yelp history,
# including the val/test reviews we hold out — using it here leaked the
# held-out target into the user feature.


#### User Tower - feature 2: review_count_log

In [6]:
# review_count_log is now computed AFTER the temporal split from train rows
# only (see post-split cell). Raw user JSON's `review_count` is a full-history
# snapshot that includes the held-out val/test reviews — same leak path as
# avg_stars_given.

import numpy as np


#### User Tower - feature 3: yelping_since_years

In [7]:
# yelping_since is a date string that represents when the user joined Yelp.

reference_date = pd.to_datetime(yelp_users_df["yelping_since"]).max()

yelp_users_df["yelping_since_years"] = (
    (reference_date - pd.to_datetime(yelp_users_df["yelping_since"])).dt.days / 365.25
)

print(f"Reference date: {reference_date}")
print(yelp_users_df["yelping_since_years"].describe())
print(f"\nSample values:\n{yelp_users_df[['user_id', 'yelping_since', 'yelping_since_years']].head(10).to_string(index=False)}")

Reference date: 2022-01-17 15:25:41
count    166488.000000
mean          8.254171
std           2.947440
min           0.000000
25%           6.220397
50%           8.325804
75%          10.417522
max          17.264887
Name: yelping_since_years, dtype: float64

Sample values:
               user_id       yelping_since  yelping_since_years
qVc8ODYU5SZjKXVBgXdI7w 2007-01-25 16:47:26            14.976044
j14WgRoU_-2ZE1aw1dXrJg 2009-01-25 04:35:42            12.977413
2WnXYQFK0hXEoTxPtV2zvg 2008-07-25 10:41:00            13.481177
SZDeASXq7o05mMNLshsdIA 2005-11-29 04:38:33            16.134155
hA5lMy-EnncsH4JoR-hFGQ 2007-01-05 19:40:59            15.030801
q_QQ5kBBwlCcbL1s4NVK3g 2005-03-14 20:26:35            16.843258
cxuxXkcihfCbqt5Byrup8Q 2009-02-24 03:09:06            12.895277
AUi8MPWJ0mLkMfwbui27lg 2010-01-07 18:32:04            12.024641
SgiBkhXeqIKl1PlFpZOycQ 2006-08-25 16:47:25            15.394935
QF1Kuhs8iwLWANNZxebTow 2009-04-27 20:25:54            12.722793


#### User Tower - feature 4: is_elite

In [8]:
# is_elite flag indicates if a user is of elite status or not.
# elite users are recognized by Yelp for writing very high-quality reviews,
# so can be considered more trustworthy + engaged reviewers

yelp_users_df["is_elite"] = (yelp_users_df["elite"].str.strip() != "").astype(int)

print(yelp_users_df["is_elite"].value_counts())
print(f"\nElite rate: {yelp_users_df['is_elite'].mean():.2%}")

is_elite
0    131238
1     35250
Name: count, dtype: int64

Elite rate: 21.17%


#### User Tower - feature 5: avg_useful_per_review

In [9]:
# avg_useful_per_review is now computed AFTER the temporal split from train
# rows only (see post-split cell). Raw user JSON's `useful` and `review_count`
# are both full-history snapshots — same leak path as avg_stars_given.


#### User Tower - feature 6: social_activity

In [10]:
# social_activity_log combines fans + friend counts to determine how socially active a user is on Yelp (and then log-scales it)

friend_count = yelp_users_df["friends"].str.split(",").str.len().fillna(0).astype(int)
yelp_users_df["social_activity"] = yelp_users_df["fans"] + friend_count
yelp_users_df["social_activity_log"] = np.log1p(yelp_users_df["social_activity"]) # log-scaling for range compression

print(yelp_users_df["social_activity_log"].describe())
print(f"\nSample values:\n{yelp_users_df[['user_id', 'fans', 'social_activity', 'social_activity_log']].head(10).to_string(index=False)}")

count    166488.000000
mean          2.988202
std           1.916192
min           0.693147
25%           1.098612
50%           2.944439
75%           4.615121
max           9.808242
Name: social_activity_log, dtype: float64

Sample values:
               user_id  fans  social_activity  social_activity_log
qVc8ODYU5SZjKXVBgXdI7w   267            15262             9.633187
j14WgRoU_-2ZE1aw1dXrJg  3138             7784             8.959954
2WnXYQFK0hXEoTxPtV2zvg    52              433             6.073045
SZDeASXq7o05mMNLshsdIA    28              159             5.075174
hA5lMy-EnncsH4JoR-hFGQ     1               28             3.367296
q_QQ5kBBwlCcbL1s4NVK3g  1357             7200             8.881975
cxuxXkcihfCbqt5Byrup8Q     1               24             3.218876
AUi8MPWJ0mLkMfwbui27lg     4               68             4.234107
SgiBkhXeqIKl1PlFpZOycQ    44              231             5.446737
QF1Kuhs8iwLWANNZxebTow   131              618             6.428105


#### User Tower - feature 7: user_price_preference

This feature is the average price range of restaurants the user has reviewed.
**Computed AFTER the temporal split** (see the cell after Train/Val/Test Splits)
so that val/test items don't leak into the user feature. Only the markdown
header lives here; the implementation is post-split.


In [11]:
# user_price_preference is computed AFTER the temporal split to avoid
# leaking val/test items into the user feature. See the post-split cell.
# (The previous implementation here averaged price_range over the full review
# table, which included future val/test reviews — a leak.)


--

#### User & Business Tower feature: categories_embedding

In [12]:
yelp_food_taxonomy = {
    "Food", "Restaurant",
    # Food
    "Acai Bowls", "Backshop", "Bagels", "Bakeries", "Beer, Wine & Spirits", "Bento",
    "Beverage Store", "Breweries", "Brewpubs", "Bubble Tea", "Butcher", "Chimney Cakes",
    "Churros", "Cideries", "Coffee & Tea", "Coffee & Tea Supplies", "Coffee Roasteries",
    "Convenience Stores", "CSA", "Cupcakes", "Custom Cakes", "Delicatessen", "Desserts",
    "Distilleries", "Do-It-Yourself Food", "Donairs", "Donuts", "Empanadas",
    "Ethical Grocery", "Farmers Market", "Fishmonger", "Food Delivery Services",
    "Food Trucks", "Friterie", "Gelato", "Grocery", "Hawker Centre", "Honey",
    "Ice Cream & Frozen Yogurt", "Imported Food", "International Grocery",
    "Internet Cafes", "Japanese Sweets", "Taiyaki", "Juice Bars & Smoothies", "Kiosk",
    "Kombucha", "Meaderies", "Milkshake Bars", "Mulled Wine", "Nasi Lemak",
    "Organic Stores", "Panzerotti", "Parent Cafes", "Patisserie/Cake Shop", "Piadina",
    "Poke", "Pretzels", "Salumerie", "Shaved Ice", "Shaved Snow", "Smokehouse",
    "Specialty Food", "Candy Stores", "Cheese Shops", "Chocolatiers & Shops", "Dagashi",
    "Dried Fruit", "Frozen Food", "Fruits & Veggies", "Health Markets", "Herbs & Spices",
    "Macarons", "Meat Shops", "Olive Oil", "Pasta Shops", "Popcorn Shops",
    "Seafood Markets", "Tofu Shops", "Street Vendors", "Sugar Shacks", "Tea Rooms",
    "Torshi", "Tortillas", "Water Stores", "Wineries", "Zapiekanka",
    # Restaurants
    "Afghan", "African", "Senegalese", "South African", "American (New)",
    "American (Traditional)", "Andalusian", "Arabian", "Arab Pizza", "Argentine",
    "Armenian", "Asian Fusion", "Asturian", "Australian", "Austrian", "Baguettes",
    "Bangladeshi", "Barbeque", "Basque", "Bavarian", "Beer Garden", "Beer Hall",
    "Beisl", "Belgian", "Flemish", "Bistros", "Black Sea", "Brasseries", "Brazilian",
    "Brazilian Empanadas", "Central Brazilian", "Northeastern Brazilian",
    "Northern Brazilian", "Rodizios", "Breakfast & Brunch", "Pancakes", "British",
    "Buffets", "Bulgarian", "Burgers", "Burmese", "Cafes", "Themed Cafes", "Cafeteria",
    "Cajun/Creole", "Cambodian", "Canadian (New)", "Canteen", "Caribbean", "Dominican",
    "Haitian", "Puerto Rican", "Trinidadian", "Catalan", "Cheesesteaks", "Chicken Shop",
    "Chicken Wings", "Chilean", "Chinese", "Cantonese", "Congee", "Dim Sum", "Fuzhou",
    "Hainan", "Hakka", "Henghwa", "Hokkien", "Hunan", "Pekinese", "Shanghainese",
    "Szechuan", "Teochew", "Comfort Food", "Corsican", "Creperies", "Cuban",
    "Curry Sausage", "Cypriot", "Czech", "Czech/Slovakian", "Danish", "Delis", "Diners",
    "Dinner Theater", "Dumplings", "Eastern European", "Eritrean", "Ethiopian",
    "Fast Food", "Filipino", "Fischbroetchen", "Fish & Chips", "Flatbread", "Fondue",
    "Food Court", "Food Stands", "Freiduria", "French", "Alsatian", "Auvergnat",
    "Berrichon", "Bourguignon", "Mauritius", "Nicoise", "Provencal", "Reunion",
    "French Southwest", "Galician", "Game Meat", "Gastropubs", "Georgian", "German",
    "Baden", "Eastern German", "Franconian", "Hessian", "Northern German", "Palatine",
    "Rhinelandian", "Giblets", "Gluten-Free", "Greek", "Guamanian", "Halal", "Hawaiian",
    "Heuriger", "Himalayan/Nepalese", "Honduran", "Hong Kong Style Cafe", "Hot Dogs",
    "Hot Pot", "Hungarian", "Iberian", "Indian", "Indonesian", "International", "Irish",
    "Island Pub", "Israeli", "Italian", "Abruzzese", "Altoatesine", "Apulian",
    "Calabrian", "Cucina Campana", "Emilian", "Friulan", "Ligurian", "Lumbard",
    "Napoletana", "Piemonte", "Roman", "Sardinian", "Sicilian", "Tuscan", "Venetian",
    "Japanese", "Blowfish", "Conveyor Belt Sushi", "Donburi", "Gyudon", "Oyakodon",
    "Hand Rolls", "Horumon", "Izakaya", "Japanese Curry", "Kaiseki", "Kushikatsu",
    "Oden", "Okinawan", "Okonomiyaki", "Onigiri", "Ramen", "Robatayaki", "Soba",
    "Sukiyaki", "Takoyaki", "Tempura", "Teppanyaki", "Tonkatsu", "Udon", "Unagi",
    "Western Style Japanese Food", "Yakiniku", "Yakitori", "Jewish", "Kebab",
    "Kopitiam", "Korean", "Kosher", "Kurdish", "Laos", "Laotian", "Latin American",
    "Colombian", "Salvadoran", "Venezuelan", "Live/Raw Food", "Lyonnais", "Malaysian",
    "Mamak", "Nyonya", "Meatballs", "Mediterranean", "Falafel", "Mexican",
    "Eastern Mexican", "Jaliscan", "Northern Mexican", "Oaxacan", "Pueblan", "Tacos",
    "Tamales", "Yucatan", "Middle Eastern", "Egyptian", "Lebanese", "Milk Bars",
    "Modern Australian", "Modern European", "Mongolian", "Moroccan", "New Mexican Cuisine",
    "New Zealand", "Nicaraguan", "Night Food", "Nikkei", "Noodles", "Norcinerie",
    "Open Sandwiches", "Oriental", "Pakistani", "Pan Asian", "Parent Cafes", "Parma",
    "Persian/Iranian", "Peruvian", "PF/Comercial", "Pita", "Pizza", "Polish", "Pierogis",
    "Polynesian", "Pop-Up Restaurants", "Portuguese", "Alentejo", "Algarve", "Azores",
    "Beira", "Fado Houses", "Madeira", "Minho", "Ribatejo", "Tras-os-Montes", "Potatoes",
    "Poutineries", "Pub Food", "Rice", "Romanian", "Rotisserie Chicken", "Russian",
    "Salad", "Sandwiches", "Scandinavian", "Schnitzel", "Scottish", "Seafood",
    "Serbo Croatian", "Signature Cuisine", "Singaporean", "Slovakian", "Somali",
    "Soul Food", "Soup", "Southern", "Spanish", "Arroceria/Paella", "Sri Lankan",
    "Steakhouses", "Supper Clubs", "Sushi Bars", "Swabian", "Swedish", "Swiss Food",
    "Syrian", "Tabernas", "Taiwanese", "Tapas Bars", "Tapas/Small Plates", "Tavola Calda",
    "Tex-Mex", "Thai", "Traditional Norwegian", "Traditional Swedish", "Trattorie",
    "Turkish", "Chee Kufta", "Gozleme", "Homemade Food", "Lahmacun", "Ottoman Cuisine",
    "Turkish Ravioli", "Ukrainian", "Uzbek", "Vegan", "Vegetarian", "Venison",
    "Vietnamese", "Waffles", "Wok", "Wraps", "Yugoslav", "Ethnic Food",
    # Bars & Nightlife
    "Bars", "Airport Lounges", "Beer Bar", "Champagne Bars", "Cigar Bars",
    "Cocktail Bars", "Dive Bars", "Drive-Thru Bars", "Gay Bars", "Hookah Bars",
    "Irish Pub", "Lounges", "Pubs", "Speakeasies", "Sports Bars", "Tiki Bars",
    "Vermouth Bars", "Whiskey Bars", "Wine Bars", "Beer Gardens", "Jazz & Blues",
    "Karaoke", "Music Venues", "Piano Bars", "Nightlife", "Beer", "Wine & Spirits",
    "Wine Tasting Room",
    # Hotels & Travel (food-adjacent)
    "Bed & Breakfast", "Hotels", "Mountain Huts", "Residences", "Rest Stops",
}

In [13]:
from sentence_transformers import SentenceTransformer

def clean_categories(cat_str):
    if pd.isna(cat_str):
        return ""
    return ", ".join(
        c.strip() for c in cat_str.split(",")
        if c.strip() in yelp_food_taxonomy
    )

yelp_open_restaurants_df["categories_clean"] = yelp_open_restaurants_df["categories"].apply(clean_categories)

# Encode each business's categories string into a 1024-dim Qwen embedding.
# Business-side only here. The user-side category embedding used to live in
# this cell but was being averaged over yelp_reviews_df (i.e. all reviews,
# including future val/test items) — that's a feature leak. The user-side
# aggregation now happens AFTER the temporal split below, restricted to
# train-positive items only.
cat_encoder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

business_cat_strings = yelp_open_restaurants_df["categories_clean"].tolist()
business_cat_embeddings = cat_encoder.encode(
    business_cat_strings, batch_size=256, show_progress_bar=True
)
yelp_open_restaurants_df["categories_embedding"] = list(business_cat_embeddings)

business_id_to_embedding = dict(zip(
    yelp_open_restaurants_df["business_id"],
    yelp_open_restaurants_df["categories_embedding"]
))

print(f"Business embedding shape: {business_cat_embeddings.shape}")
print(f"Sample business embedding (first 8 dims): {business_cat_embeddings[0][:8]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/175 [00:00<?, ?it/s]

Business embedding shape: (44582, 1024)
Sample business embedding (first 8 dims): [-0.05883789 -0.01733398 -0.01123047 -0.01080322 -0.03295898 -0.07080078
  0.09033203  0.0559082 ]


--

#### Business Tower - feature 1: avg_stars

In [14]:
# Average stars for open restaurant businesses
yelp_open_restaurants_df["avg_stars"] = yelp_open_restaurants_df["stars"]
print(yelp_open_restaurants_df["avg_stars"].describe())
print(f"\nSample values:\n{yelp_open_restaurants_df[['business_id', 'avg_stars']].head(10).to_string(index=False)}")

count    44582.000000
mean         3.548506
std          0.888338
min          1.000000
25%          3.000000
50%          3.500000
75%          4.000000
max          5.000000
Name: avg_stars, dtype: float64

Sample values:
           business_id  avg_stars
MTSW4McQd7CbVtyjqoe9mw        4.0
mWMc6_wTdE0EUBKIGXDVfA        4.5
CF33F8-E6oudUQ46HnavjQ        2.0
bBDDEgkFA1Otx9Lfe7BZUQ        1.5
eEOYSgkmpB90uNA7lDOMRA        4.0
il_Ro8jwPlHresjw9EGmBg        2.5
MUTTqe8uqyMdBl186RmNeA        4.0
ROeacJQwBeh05Rqg7F6TCg        4.5
kfNv-JZpuN6TVNSO6hHdkw        4.0
9OG5YkX1g2GReZM0AskizA        2.5


#### Business Tower - feature 2: review_count_log

In [15]:
# Log-scaled review counts for open restaurant businesses
yelp_open_restaurants_df["review_count_log"] = np.log1p(yelp_open_restaurants_df["review_count"])
print(yelp_open_restaurants_df[["review_count", "review_count_log"]].describe())
print(f"\nSample values:\n{yelp_open_restaurants_df[['business_id', 'review_count', 'review_count_log']].head(10).to_string(index=False)}")

       review_count  review_count_log
count  44582.000000      44582.000000
mean      88.924745          3.630020
std      199.810367          1.232965
min        5.000000          1.791759
25%       13.000000          2.639057
50%       32.000000          3.496508
75%       88.000000          4.488636
max     7568.000000          8.931816

Sample values:
           business_id  review_count  review_count_log
MTSW4McQd7CbVtyjqoe9mw            80          4.394449
mWMc6_wTdE0EUBKIGXDVfA            13          2.639057
CF33F8-E6oudUQ46HnavjQ             6          1.945910
bBDDEgkFA1Otx9Lfe7BZUQ            10          2.397895
eEOYSgkmpB90uNA7lDOMRA            10          2.397895
il_Ro8jwPlHresjw9EGmBg            28          3.367296
MUTTqe8uqyMdBl186RmNeA           245          5.505332
ROeacJQwBeh05Rqg7F6TCg           205          5.327876
kfNv-JZpuN6TVNSO6hHdkw            20          3.044522
9OG5YkX1g2GReZM0AskizA           339          5.828946


#### Business Tower - feature 3: price_range

In [16]:
# similar logic to user_price_preference
yelp_open_restaurants_df["price_range"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: int(x["RestaurantsPriceRange2"])
    if isinstance(x, dict) and x.get("RestaurantsPriceRange2") not in [None, "None"]
    else None
)

#### Business Tower - feature 4: city_encoded and state_encoded

In [17]:
from sklearn.preprocessing import LabelEncoder

# Label encoding for city and state
city_encoder = LabelEncoder()
state_encoder = LabelEncoder()
yelp_open_restaurants_df["city_encoded"] = city_encoder.fit_transform(
    yelp_open_restaurants_df["city"].fillna("Unknown")
)
yelp_open_restaurants_df["state_encoded"] = state_encoder.fit_transform(
    yelp_open_restaurants_df["state"].fillna("Unknown")
)

print(f"Unique cities: {yelp_open_restaurants_df['city_encoded'].nunique()}")
print(f"Unique states: {yelp_open_restaurants_df['state_encoded'].nunique()}")
print(f"\nSample values:\n{yelp_open_restaurants_df[['business_id', 'city', 'city_encoded', 'state', 'state_encoded']].head(10).to_string(index=False)}")

Unique cities: 919
Unique states: 16

Sample values:
           business_id         city  city_encoded state  state_encoded
MTSW4McQd7CbVtyjqoe9mw Philadelphia           574    PA             13
mWMc6_wTdE0EUBKIGXDVfA   Green Lane           289    PA             13
CF33F8-E6oudUQ46HnavjQ Ashland City            16    TN             14
bBDDEgkFA1Otx9Lfe7BZUQ    Nashville           502    TN             14
eEOYSgkmpB90uNA7lDOMRA    Tampa Bay           761    FL              4
il_Ro8jwPlHresjw9EGmBg Indianapolis           344    IN              8
MUTTqe8uqyMdBl186RmNeA Philadelphia           574    PA             13
ROeacJQwBeh05Rqg7F6TCg Philadelphia           574    PA             13
kfNv-JZpuN6TVNSO6hHdkw Indianapolis           344    IN              8
9OG5YkX1g2GReZM0AskizA         Reno           613    NV             12


#### Business Tower - feature 5: ambience flags

In [18]:
import ast

def parse_attr_dict(val):
    if val is None or val == "None" or not isinstance(val, str):
        return {}
    try:
        return ast.literal_eval(val.replace("u'", "'").replace('u"', '"'))
    except:
        return {}

def parse_bool_attr(val):
    if val is None or val == "None":
        return None
    return 1 if str(val).strip().lower() == "true" else 0

# helps for telling the model the kind of atmosphere or vibe each restaurant has
ambience_flags = ["romantic", "intimate", "classy", "hipster", "divey",
                  "touristy", "trendy", "upscale", "casual"]

ambience_parsed = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_attr_dict(x.get("Ambience")) if isinstance(x, dict) else {}
)

for flag in ambience_flags:
    yelp_open_restaurants_df[f"ambience_{flag}"] = ambience_parsed.apply(
        lambda d: int(d[flag]) if flag in d and isinstance(d[flag], bool) else None
    )

print(yelp_open_restaurants_df[[f"ambience_{f}" for f in ambience_flags]].describe())
print(f"\nNull counts:\n{yelp_open_restaurants_df[[f'ambience_{f}' for f in ambience_flags]].isna().sum()}")

       ambience_romantic  ambience_intimate  ambience_classy  \
count       24566.000000       23805.000000     24333.000000   
mean            0.017056           0.020500         0.194222   
std             0.129483           0.141706         0.395609   
min             0.000000           0.000000         0.000000   
25%             0.000000           0.000000         0.000000   
50%             0.000000           0.000000         0.000000   
75%             0.000000           0.000000         0.000000   
max             1.000000           1.000000         1.000000   

       ambience_hipster  ambience_divey  ambience_touristy  ambience_trendy  \
count      23745.000000    22916.000000       24334.000000     22871.000000   
mean           0.029143        0.030372           0.009329         0.072013   
std            0.168211        0.171612           0.096135         0.258514   
min            0.000000        0.000000           0.000000         0.000000   
25%            0.000000     

#### Business Tower - feature 6: noise_level

In [19]:
# noise level feature is used for matching user preferences based on noise of a restaurant
noise_map = {"quiet": 0, "average": 1, "loud": 2, "very_loud": 3}

def parse_noise(val):
    if val is None or val == "None":
        return None
    cleaned = str(val).replace("u'", "").replace("'", "").strip().lower()
    return noise_map.get(cleaned, None)

yelp_open_restaurants_df["noise_level"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_noise(x.get("NoiseLevel")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["noise_level"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['noise_level'].isna().sum()}")

noise_level
1.0    16905
0.0     4001
2.0     1660
3.0      506
Name: count, dtype: int64
Nulls: 21510


#### Business Tower - feature 7: good_for_meal flag

In [20]:
# good for meals tells which meal periods a restaurant is good for, so
# there are six particular flags here and helps for matching users
# based on when they tend to eat out

meal_flags = ["breakfast", "brunch", "lunch", "dinner", "latenight", "dessert"]

good_for_meal_parsed = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_attr_dict(x.get("GoodForMeal")) if isinstance(x, dict) else {}
)

for flag in meal_flags:
    yelp_open_restaurants_df[f"good_for_meal_{flag}"] = good_for_meal_parsed.apply(
        lambda d: int(d[flag]) if flag in d and isinstance(d[flag], bool) else None
    )

print(yelp_open_restaurants_df[[f"good_for_meal_{f}" for f in meal_flags]].describe())
print(f"\nNull counts:\n{yelp_open_restaurants_df[[f'good_for_meal_{f}' for f in meal_flags]].isna().sum()}")

       good_for_meal_breakfast  good_for_meal_brunch  good_for_meal_lunch  \
count             19259.000000          18196.000000         19302.000000   
mean                  0.122644              0.116619             0.564708   
std                   0.328037              0.320975             0.495808   
min                   0.000000              0.000000             0.000000   
25%                   0.000000              0.000000             0.000000   
50%                   0.000000              0.000000             1.000000   
75%                   0.000000              0.000000             1.000000   
max                   1.000000              1.000000             1.000000   

       good_for_meal_dinner  good_for_meal_latenight  good_for_meal_dessert  
count          20257.000000             18080.000000           17289.000000  
mean               0.506738                 0.058960               0.080514  
std                0.499967                 0.235557               0.272

#### Business Tower - feature 8: alcohol

In [21]:
# alcohol feature represents level of alcohol service at the restaurant
alcohol_map = {"none": 0, "beer_and_wine": 1, "full_bar": 2}

def parse_alcohol(val):
    if val is None or val == "None":
        return None
    cleaned = str(val).replace("u'", "").replace("'", "").strip().lower()
    return alcohol_map.get(cleaned, None)

yelp_open_restaurants_df["alcohol"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_alcohol(x.get("Alcohol")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["alcohol"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['alcohol'].isna().sum()}")

alcohol
0.0    13995
2.0     8544
1.0     3827
Name: count, dtype: int64
Nulls: 18216


#### Business Tower - feature 9: outdoor_seating

In [22]:
# helps determine restaurants with outdoor seating for outdoor seating filtering downstream

yelp_open_restaurants_df["outdoor_seating"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("OutdoorSeating")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["outdoor_seating"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['outdoor_seating'].isna().sum()}")

outdoor_seating
0.0    14667
1.0    13784
Name: count, dtype: int64
Nulls: 16131


#### Business Tower - feature 10: takes_reservations

In [23]:
# takes_reservations helps for determining if a restaurant takes reservations for our recommender

yelp_open_restaurants_df["takes_reservations"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("RestaurantsReservations")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["takes_reservations"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['takes_reservations'].isna().sum()}")

takes_reservations
0.0    19913
1.0     8385
Name: count, dtype: int64
Nulls: 16284


#### Business Tower - feature 11: Restaurant Take-Out

In [24]:
# restaurant take out helps for recommendations if a take out filter is used
yelp_open_restaurants_df["restaurants_take_out"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("RestaurantsTakeOut")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["restaurants_take_out"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['restaurants_take_out'].isna().sum()}")

restaurants_take_out
1.0    36097
0.0     3212
Name: count, dtype: int64
Nulls: 5273


#### Business Tower - feature 12: restaurants_delivery

In [25]:
# restaurants delivery is useful for recommending restaurants that have delivery as an option
yelp_open_restaurants_df["restaurants_delivery"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("RestaurantsDelivery")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["restaurants_delivery"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['restaurants_delivery'].isna().sum()}")

restaurants_delivery
1.0    25990
0.0     9692
Name: count, dtype: int64
Nulls: 8900


#### Business Tower - feature 13: good_for_kids

In [26]:
# good for kids is a feature that's helpful for family-friendly recommendations
yelp_open_restaurants_df["good_for_kids"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("GoodForKids")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["good_for_kids"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['good_for_kids'].isna().sum()}")

good_for_kids
1.0    23274
0.0     3599
Name: count, dtype: int64
Nulls: 17709


#### Business Tower - feature 14: good_for_groups

In [27]:
# good for groups is useful for recommendations of restaurants good to visit as a group
yelp_open_restaurants_df["good_for_groups"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_bool_attr(x.get("RestaurantsGoodForGroups")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["good_for_groups"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['good_for_groups'].isna().sum()}")

good_for_groups
1.0    22695
0.0     3963
Name: count, dtype: int64
Nulls: 17924


#### Business Tower - feature 15: attire

In [28]:
# attire feature helps with understanding expected dress code (capture formality of a dining experience)
attire_map = {"casual": 0, "dressy": 1, "formal": 2}

def parse_attire(val):
    if val is None or val == "None":
        return None
    cleaned = str(val).replace("u'", "").replace("'", "").strip().lower()
    return attire_map.get(cleaned, None)

yelp_open_restaurants_df["attire"] = yelp_open_restaurants_df["attributes"].apply(
    lambda x: parse_attire(x.get("RestaurantsAttire")) if isinstance(x, dict) else None
)

print(yelp_open_restaurants_df["attire"].value_counts())
print(f"Nulls: {yelp_open_restaurants_df['attire'].isna().sum()}")

attire
0.0    23972
1.0      425
2.0       50
Name: count, dtype: int64
Nulls: 20135


### Train, Validation, Test Splits

In [29]:
yelp_reviews_df["date"] = pd.to_datetime(yelp_reviews_df["date"])
yelp_reviews_df = yelp_reviews_df.sort_values(["user_id", "date"])
yelp_reviews_df["user_review_rank"] = yelp_reviews_df.groupby("user_id").cumcount(ascending=False)

# Temporal leave-last-out: rank 0 = newest (test), rank 1 = val, rest = train.
test_df  = yelp_reviews_df[yelp_reviews_df["user_review_rank"] == 0].copy()
val_df   = yelp_reviews_df[yelp_reviews_df["user_review_rank"] == 1].copy()
train_df = yelp_reviews_df[yelp_reviews_df["user_review_rank"] >= 2].copy()

# Absolute-threshold label: stars >= 4 -> positive. Standard in the recsys
# literature (NCF, LightGCN, SASRec). Replaces the prior `stars > user_mean`
# scheme which was incoherent across users (a 3-star review was positive for
# a 2-star-mean user, negative for a 4-star-mean user). Because we no longer
# derive user_mean at all, the user_mean leak is structurally impossible.
def relabel(df, name):
    df = df.copy()
    df["label"] = (df["stars"] >= 4).astype(np.float32)
    print(f"  {name}: {len(df):,} rows | positive rate {df['label'].mean():.4%}")
    return df

train_df = relabel(train_df, "train")
val_df   = relabel(val_df,   "val")
test_df  = relabel(test_df,  "test")

print(f"\nTrain: {len(train_df)} reviews")
print(f"Val:   {len(val_df)} reviews")
print(f"Test:  {len(test_df)} reviews")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} reviews")


  train: 1,990,604 rows | positive rate 70.9126%
  val: 166,488 rows | positive rate 70.7516%
  test: 166,488 rows | positive rate 70.0915%

Train: 1990604 reviews
Val:   166488 reviews
Test:  166488 reviews
Total: 2323580 reviews


In [30]:
# Per-user category preference embedding from train-positive reviews only.
# This is the user-side counterpart to the business Qwen embedding above.
# Restricting to train-positives prevents the user feature from encoding
# val/test item identity — the dominant data leak in the original notebook.
train_pos = train_df[train_df["label"] == 1]
emb_dim   = business_cat_embeddings.shape[1]

biz_index_map = {b: i for i, b in enumerate(yelp_open_restaurants_df["business_id"].values)}
biz_emb_array = np.stack(yelp_open_restaurants_df["categories_embedding"].values)

mask = train_pos["business_id"].isin(biz_index_map)
tp   = train_pos[mask]
biz_rows = tp["business_id"].map(biz_index_map).to_numpy(dtype=np.int64)

# Vectorized per-user mean: factorize user_ids, sum embeddings via np.add.at.
user_codes, users_arr = pd.factorize(tp["user_id"], sort=True)
sums   = np.zeros((len(users_arr), emb_dim), dtype=np.float32)
counts = np.zeros(len(users_arr), dtype=np.int64)
np.add.at(sums, user_codes, biz_emb_array[biz_rows])
np.add.at(counts, user_codes, 1)
user_cat_emb_arr = sums / np.maximum(counts, 1)[:, None]

user_cat_embeddings = pd.DataFrame({
    "user_id": users_arr,
    "categories_embedding": list(user_cat_emb_arr),
})

if "categories_embedding" in yelp_users_df.columns:
    yelp_users_df = yelp_users_df.drop(columns=["categories_embedding"])

# Inner-merge: users without train positives are dropped (cold-start guard).
yelp_users_df = yelp_users_df.merge(user_cat_embeddings, on="user_id", how="inner")

print(f"User embedding shape: {user_cat_emb_arr.shape}")
print(f"Sample user embedding (first 8 dims): {yelp_users_df['categories_embedding'].iloc[0][:8]}")


User embedding shape: (162366, 1024)
Sample user embedding (first 8 dims): [-0.03499756 -0.00881348 -0.0107605  -0.01120911  0.03851929 -0.06845703
  0.02097168  0.03250122]


In [31]:
# user_price_preference computed from train rows only (LEAK FIX).
# Was previously computed over the full review table including future val/test.
# Now: average price_range of restaurants the user reviewed in TRAIN.
# Cold-start users (no train rows after the upstream filters) get the train-
# population mean as a fallback.
price_map = yelp_open_restaurants_df[["business_id", "attributes"]].copy()
price_map["price_range"] = price_map["attributes"].apply(
    lambda x: int(x["RestaurantsPriceRange2"])
    if isinstance(x, dict) and x.get("RestaurantsPriceRange2") not in ["None", None]
    else None
)

user_price_pref_df = (
    train_df[["user_id", "business_id"]]
        .merge(price_map[["business_id", "price_range"]], on="business_id", how="left")
        .groupby("user_id")["price_range"].mean()
        .reset_index()
        .rename(columns={"price_range": "user_price_preference"})
)

# Drop any pre-existing (leaky) column from earlier runs before merging.
if "user_price_preference" in yelp_users_df.columns:
    yelp_users_df = yelp_users_df.drop(columns=["user_price_preference"])

yelp_users_df = yelp_users_df.merge(user_price_pref_df, on="user_id", how="left")

# Train-population mean as cold-start fallback (avoids leaking val/test prices).
train_pop_mean = user_price_pref_df["user_price_preference"].mean()
n_filled = yelp_users_df["user_price_preference"].isna().sum()
yelp_users_df["user_price_preference"] = yelp_users_df["user_price_preference"].fillna(train_pop_mean)

print(yelp_users_df["user_price_preference"].describe())
print(f"Filled {n_filled} NaNs with train-population mean ({train_pop_mean:.4f})")
print(f"Nulls remaining: {yelp_users_df['user_price_preference'].isna().sum()}")


count    162366.000000
mean          1.860814
std           0.288972
min           1.000000
25%           1.666667
50%           1.857143
75%           2.000000
max           4.000000
Name: user_price_preference, dtype: float64
Filled 18 NaNs with train-population mean (1.8581)
Nulls remaining: 0


In [32]:
# Train-only recompute of three user features that previously copied raw
# Yelp user JSON aggregates. Those aggregates are dataset-extraction-time
# snapshots that include the val/test reviews held out below — a feature
# leak. Pattern mirrors user_price_preference above.
#
# Verified with /tests/ scripts (deleted after): avg_stars_given carried
# corr ~0.47 with the held-out test star residual; recomputing closes that
# channel.

train_user_stats = (
    train_df.groupby("user_id")
    .agg(
        avg_stars_given=("stars", "mean"),
        review_count_train=("stars", "count"),
        useful_sum=("useful", "sum"),
    )
    .reset_index()
)
train_user_stats["review_count_log"]      = np.log1p(train_user_stats["review_count_train"])
train_user_stats["avg_useful_per_review"] = train_user_stats["useful_sum"] / train_user_stats["review_count_train"]

# Drop leaky originals before merging the train-only versions in.
for col in ["avg_stars_given", "review_count_log", "avg_useful_per_review"]:
    if col in yelp_users_df.columns:
        yelp_users_df = yelp_users_df.drop(columns=[col])

yelp_users_df = yelp_users_df.merge(
    train_user_stats[["user_id", "avg_stars_given", "review_count_log", "avg_useful_per_review"]],
    on="user_id",
    how="left",
)

# Cold-start fallback: train-population means. After k-core (k=5) every user
# has >=3 train rows, so this should rarely trigger — defensive only.
pop_avg_stars   = train_user_stats["avg_stars_given"].mean()
pop_count_log   = train_user_stats["review_count_log"].mean()
pop_useful_rate = train_user_stats["avg_useful_per_review"].mean()

n_cold = yelp_users_df["avg_stars_given"].isna().sum()
yelp_users_df["avg_stars_given"]      = yelp_users_df["avg_stars_given"].fillna(pop_avg_stars)
yelp_users_df["review_count_log"]     = yelp_users_df["review_count_log"].fillna(pop_count_log)
yelp_users_df["avg_useful_per_review"]= yelp_users_df["avg_useful_per_review"].fillna(pop_useful_rate)

print(f"Train-only recompute: filled {n_cold} cold-start users with population means")
print(f"  pop_avg_stars={pop_avg_stars:.4f}  pop_count_log={pop_count_log:.4f}  pop_useful_rate={pop_useful_rate:.4f}")
print(yelp_users_df[["avg_stars_given", "review_count_log", "avg_useful_per_review"]].describe())


Train-only recompute: filled 0 cold-start users with population means
  pop_avg_stars=3.8852  pop_count_log=2.1346  pop_useful_rate=0.8403
       avg_stars_given  review_count_log  avg_useful_per_review
count    162366.000000     162366.000000          162366.000000
mean          3.941711          2.148859               0.841775
std           0.696060          0.767223               1.640926
min           1.205882          1.386294               0.000000
25%           3.523810          1.609438               0.200000
50%           4.000000          1.945910               0.500000
75%           4.444444          2.484907               1.000000
max           5.000000          7.266129             161.333333


### Save Data

In [33]:
import os
import numpy as np

save_dir = "/content/drive/MyDrive/rec_system/newV2"
os.makedirs(save_dir, exist_ok=True)

# temporary helper column used for performing the splits
train_df.drop(columns=["user_review_rank"]).to_parquet(f"{save_dir}/train_reviews.parquet", index=False)
val_df.drop(columns=["user_review_rank"]).to_parquet(f"{save_dir}/val_reviews.parquet", index=False)
test_df.drop(columns=["user_review_rank"]).to_parquet(f"{save_dir}/test_reviews.parquet", index=False)

# the user and business scalar cols save those features to parquet and includes the engineered attributes
# user_id and business_id are also included as join keys for aligning with embeddings
user_scalar_cols = [
    "user_id", "avg_stars_given", "review_count_log", "yelping_since_years",
    "is_elite", "avg_useful_per_review", "social_activity_log", "user_price_preference"
]
yelp_users_df[user_scalar_cols].to_parquet(f"{save_dir}/users.parquet", index=False)

business_scalar_cols = [
    "business_id", "avg_stars", "review_count_log", "price_range",
    "city_encoded", "state_encoded", "noise_level", "alcohol",
    "outdoor_seating", "takes_reservations", "restaurants_take_out",
    "restaurants_delivery", "good_for_kids", "good_for_groups", "attire"
] + [f"ambience_{f}" for f in ["romantic", "intimate", "classy", "hipster", "divey", "touristy", "trendy", "upscale", "casual"]] \
  + [f"good_for_meal_{f}" for f in ["breakfast", "brunch", "lunch", "dinner", "latenight", "dessert"]]
yelp_open_restaurants_df[business_scalar_cols].to_parquet(f"{save_dir}/businesses.parquet", index=False)

# embeddings are stored as .npy files to handle array columns better (and will be used later)
user_embeddings = np.stack(yelp_users_df["categories_embedding"].values)
business_embeddings = np.stack(yelp_open_restaurants_df["categories_embedding"].values)

np.save(f"{save_dir}/user_category_embeddings.npy", user_embeddings)
np.save(f"{save_dir}/business_category_embeddings.npy", business_embeddings)

yelp_users_df[["user_id"]].to_parquet(f"{save_dir}/user_embedding_index.parquet", index=False)
yelp_open_restaurants_df[["business_id"]].to_parquet(f"{save_dir}/business_embedding_index.parquet", index=False)

print("Saved!")
print(f"train_reviews.parquet: {len(train_df)} rows")
print(f"val_reviews.parquet: {len(val_df)} rows")
print(f"test_reviews.parquet: {len(test_df)} rows")
print(f"users.parquet: {len(yelp_users_df)} rows")
print(f"businesses.parquet: {len(yelp_open_restaurants_df)} rows")
print(f"user_category_embeddings.npy: {user_embeddings.shape}")
print(f"business_category_embeddings.npy: {business_embeddings.shape}")


Saved!
train_reviews.parquet: 1990604 rows
val_reviews.parquet: 166488 rows
test_reviews.parquet: 166488 rows
users.parquet: 162366 rows
businesses.parquet: 44582 rows
user_category_embeddings.npy: (162366, 1024)
business_category_embeddings.npy: (44582, 1024)


In [34]:
# Per-split positive rates. Threshold is absolute (stars >= 4), so the rate
# reflects the genuine fraction of high-rated reviews per split — not a
# per-user calibration artifact.
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5}: positive rate {df['label'].mean():.4%}, n={len(df):,}")


train: positive rate 70.9126%, n=1,990,604
  val: positive rate 70.7516%, n=166,488
 test: positive rate 70.0915%, n=166,488
